# 5.3 Crea las tablas con SQL Alchemy

In [5]:
!pip3 install -r requirements.txt -q

In [6]:
import logging
import boto3
import json
import pandas as pd
from botocore.exceptions import ClientError

In [7]:
# ──────────────────────────────────────────────
# Configuración del logger
# ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

In [8]:
def get_secret():

    secret_name = "itam/rds/flights/credentials"
    region_name = "us-east-1"

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        logger.error(f"Error retrieving secret: {e}")
        raise e

    secret = get_secret_value_response['SecretString']
    return json.loads(secret)

creds   = get_secret()

In [9]:
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

RDS_ENDPOINT = "itam-flights-150215480648.c63uaymyow80.us-east-1.rds.amazonaws.com"
DB_NAME      = creds["dbname"]
DB_USER      = creds["username"]
DB_PASSWORD  = creds["password"]
DB_PORT      = creds["port"]

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{RDS_ENDPOINT}:{DB_PORT}/{DB_NAME}"
)

# Verificar conexión
try:
    with engine.connect() as conn:
        print("✓ Conexión exitosa a RDS PostgreSQL")
except SQLAlchemyError as e:
    print(f"✗ Error de conexión: {e}")

✓ Conexión exitosa a RDS PostgreSQL


###  Define las tres tablas (airlines, airports, flights) respetando las relaciones del ERD que dibujaste en el paso anterior. Conecta al endpoint primario (RdsEndpoint) — las escrituras van siempre a la primaria.



In [10]:
from IPython.display import HTML

HTML("""
<div style="font-family: monospace; background: #0f1117; padding: 32px; border-radius: 12px; display: inline-block; width: 860px;">

  <div style="color:#8b9cf4; font-size:13px; font-weight:bold; letter-spacing:2px; margin-bottom:20px;">
    ● ENTITY RELATIONSHIP DIAGRAM — flights
  </div>

  <div style="position:relative; width:820px; height:520px;">

    <!-- SVG relationship lines -->
    <svg style="position:absolute;top:0;left:0;width:100%;height:100%;overflow:visible;" xmlns="http://www.w3.org/2000/svg">
      <!-- flights.airline → airlines.iata_code -->
      <line x1="310" y1="148" x2="490" y2="80"  stroke="#8b9cf4" stroke-width="1.5" stroke-dasharray="5,3"/>
      <circle cx="310" cy="148" r="3" fill="#8b9cf4"/>
      <circle cx="490" cy="80"  r="3" fill="#8b9cf4"/>

      <!-- flights.origin_airport → airports.iata_code -->
      <line x1="310" y1="186" x2="490" y2="320" stroke="#34d399" stroke-width="1.5" stroke-dasharray="5,3"/>
      <circle cx="310" cy="186" r="3" fill="#34d399"/>
      <circle cx="490" cy="320" r="3" fill="#34d399"/>

      <!-- flights.destination_airport → airports.iata_code -->
      <line x1="310" y1="210" x2="490" y2="334" stroke="#f472b6" stroke-width="1.5" stroke-dasharray="5,3"/>
      <circle cx="310" cy="210" r="3" fill="#f472b6"/>
      <circle cx="490" cy="334" r="3" fill="#f472b6"/>

      <!-- Cardinality labels -->
      <text x="340" y="132" fill="#8b9cf4" font-size="10" font-family="monospace">N:1</text>
      <text x="340" y="178" fill="#34d399" font-size="10" font-family="monospace">N:1</text>
      <text x="340" y="222" fill="#f472b6" font-size="10" font-family="monospace">N:1</text>
    </svg>

    <!-- FLIGHTS TABLE -->
    <div style="position:absolute;top:0;left:0;width:300px;
                background:#1a1d2e;border:1px solid #2d3154;border-radius:8px;overflow:hidden;">
      <div style="background:#2d3154;padding:8px 14px;color:#8b9cf4;font-weight:bold;font-size:12px;
                  letter-spacing:1px;">✈  flights</div>
      <div style="padding:8px 14px;font-size:11px;line-height:2;">
        <div><span style="color:#fbbf24;font-weight:bold;">PK</span>  <span style="color:#e2e8f0;">flight_id</span> <span style="color:#64748b;">INTEGER</span></div>
        <div style="color:#94a3b8;">── date ──────────────</div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">year / month / day</span> <span style="color:#64748b;">INTEGER</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">day_of_week</span> <span style="color:#64748b;">INTEGER</span></div>
        <div style="color:#94a3b8;">── references ────────</div>
        <div><span style="color:#8b9cf4;font-weight:bold;">FK</span>  <span style="color:#8b9cf4;">airline</span> <span style="color:#64748b;">VARCHAR(2)</span></div>
        <div><span style="color:#34d399;font-weight:bold;">FK</span>  <span style="color:#34d399;">origin_airport</span> <span style="color:#64748b;">VARCHAR(3)</span></div>
        <div><span style="color:#f472b6;font-weight:bold;">FK</span>  <span style="color:#f472b6;">destination_airport</span> <span style="color:#64748b;">VARCHAR(3)</span></div>
        <div style="color:#94a3b8;">── flight info ───────</div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">flight_number</span> <span style="color:#64748b;">INTEGER</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">tail_number</span> <span style="color:#64748b;">VARCHAR(10)</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">distance</span> <span style="color:#64748b;">FLOAT</span></div>
        <div style="color:#94a3b8;">── delays ────────────</div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">departure_delay</span> <span style="color:#64748b;">FLOAT</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">arrival_delay</span> <span style="color:#64748b;">FLOAT</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">air_system_delay</span> <span style="color:#64748b;">FLOAT</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">airline_delay</span> <span style="color:#64748b;">FLOAT</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">weather_delay</span> <span style="color:#64748b;">FLOAT</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">late_aircraft_delay</span> <span style="color:#64748b;">FLOAT</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">security_delay</span> <span style="color:#64748b;">FLOAT</span></div>
        <div style="color:#94a3b8;">── status ────────────</div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">cancelled</span> <span style="color:#64748b;">INTEGER</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">cancellation_reason</span> <span style="color:#64748b;">VARCHAR(1)</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">diverted</span> <span style="color:#64748b;">INTEGER</span></div>
      </div>
    </div>

    <!-- AIRLINES TABLE -->
    <div style="position:absolute;top:0;left:490px;width:240px;
                background:#1a1d2e;border:1px solid #2d3154;border-radius:8px;overflow:hidden;">
      <div style="background:#2d3154;padding:8px 14px;color:#8b9cf4;font-weight:bold;font-size:12px;
                  letter-spacing:1px;">🏢  airlines</div>
      <div style="padding:8px 14px;font-size:11px;line-height:2;">
        <div><span style="color:#fbbf24;font-weight:bold;">PK</span>  <span style="color:#8b9cf4;">iata_code</span> <span style="color:#64748b;">VARCHAR(2)</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">airline</span> <span style="color:#64748b;">VARCHAR(255)</span></div>
      </div>
    </div>

    <!-- AIRPORTS TABLE -->
    <div style="position:absolute;top:280px;left:490px;width:240px;
                background:#1a1d2e;border:1px solid #2d3154;border-radius:8px;overflow:hidden;">
      <div style="background:#2d3154;padding:8px 14px;color:#8b9cf4;font-weight:bold;font-size:12px;
                  letter-spacing:1px;">🛬  airports</div>
      <div style="padding:8px 14px;font-size:11px;line-height:2;">
        <div><span style="color:#fbbf24;font-weight:bold;">PK</span>  <span style="color:#34d399;">iata_code</span> <span style="color:#64748b;">VARCHAR(3)</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">airport</span> <span style="color:#64748b;">VARCHAR(255)</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">city</span> <span style="color:#64748b;">VARCHAR(255)</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">state</span> <span style="color:#64748b;">VARCHAR(2)</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">country</span> <span style="color:#64748b;">VARCHAR(255)</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">latitude</span> <span style="color:#64748b;">FLOAT</span></div>
        <div><span style="color:#64748b;width:24px;display:inline-block;">  </span><span style="color:#e2e8f0;">longitude</span> <span style="color:#64748b;">FLOAT</span></div>
      </div>
    </div>

    <!-- Legend -->
    <div style="position:absolute;bottom:0;left:0;font-size:10px;color:#64748b;display:flex;gap:16px;">
      <span><span style="color:#fbbf24;">PK</span> primary key</span>
      <span><span style="color:#8b9cf4;">FK</span> → airlines</span>
      <span><span style="color:#34d399;">FK</span> → airports (origin)</span>
      <span><span style="color:#f472b6;">FK</span> → airports (dest)</span>
    </div>

  </div>
</div>
""")

In [11]:
# ──────────────────────────────────────────────
# Definición de tablas con SQLAlchemy 2.0
# ──────────────────────────────────────────────
from sqlalchemy import String, Integer, Float, ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

class Base(DeclarativeBase):
    pass

class Airlines(Base):
    __tablename__ = "airlines"
    iata_code: Mapped[str] = mapped_column(String(2), primary_key=True)
    airline:   Mapped[str] = mapped_column(String(255), nullable=False)

class Airports(Base):
    __tablename__ = "airports"
    iata_code: Mapped[str]   = mapped_column(String(3), primary_key=True)
    airport:   Mapped[str]   = mapped_column(String(255), nullable=False)
    city:      Mapped[str]   = mapped_column(String(255), nullable=True)
    state:     Mapped[str]   = mapped_column(String(2), nullable=True)
    country:   Mapped[str]   = mapped_column(String(255), nullable=True)
    latitude:  Mapped[float] = mapped_column(Float, nullable=True)
    longitude: Mapped[float] = mapped_column(Float, nullable=True)

class Flights(Base):
    __tablename__ = "flights"
    
    # Llave primaria autoincremental
    flight_id:           Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    
    year:                Mapped[int] = mapped_column(Integer, nullable=False)
    month:               Mapped[int] = mapped_column(Integer, nullable=False)
    day:                 Mapped[int] = mapped_column(Integer, nullable=False)
    day_of_week:         Mapped[int] = mapped_column(Integer, nullable=True)
    airline:             Mapped[str] = mapped_column(String(2), ForeignKey("airlines.iata_code"), nullable=False)
    flight_number:       Mapped[int] = mapped_column(Integer, nullable=True)
    tail_number:         Mapped[str] = mapped_column(String(10), nullable=True)
    origin_airport:      Mapped[str] = mapped_column(String(3), ForeignKey("airports.iata_code"), nullable=False)
    destination_airport: Mapped[str] = mapped_column(String(3), ForeignKey("airports.iata_code"), nullable=False)
    scheduled_departure: Mapped[int] = mapped_column(Integer, nullable=True)
    departure_time:      Mapped[float] = mapped_column(Float, nullable=True)
    departure_delay:     Mapped[float] = mapped_column(Float, nullable=True)
    taxi_out:            Mapped[float] = mapped_column(Float, nullable=True)
    wheels_off:          Mapped[float] = mapped_column(Float, nullable=True)
    scheduled_time:      Mapped[float] = mapped_column(Float, nullable=True)
    elapsed_time:        Mapped[float] = mapped_column(Float, nullable=True)
    air_time:            Mapped[float] = mapped_column(Float, nullable=True)
    distance:            Mapped[float] = mapped_column(Float, nullable=True)
    wheels_on:           Mapped[float] = mapped_column(Float, nullable=True)
    taxi_in:             Mapped[float] = mapped_column(Float, nullable=True)
    scheduled_arrival:   Mapped[int] = mapped_column(Integer, nullable=True)
    arrival_time:        Mapped[float] = mapped_column(Float, nullable=True)
    arrival_delay:       Mapped[float] = mapped_column(Float, nullable=True)
    diverted:            Mapped[int] = mapped_column(Integer, nullable=True)
    cancelled:           Mapped[int] = mapped_column(Integer, nullable=True)
    cancellation_reason: Mapped[str] = mapped_column(String(1), nullable=True)
    air_system_delay:    Mapped[float] = mapped_column(Float, nullable=True)
    security_delay:      Mapped[float] = mapped_column(Float, nullable=True)
    airline_delay:       Mapped[float] = mapped_column(Float, nullable=True)
    late_aircraft_delay: Mapped[float] = mapped_column(Float, nullable=True)
    weather_delay:       Mapped[float] = mapped_column(Float, nullable=True)

print("✓ Tablas definidas con SQLAlchemy 2.0")

✓ Tablas definidas con SQLAlchemy 2.0


In [12]:
# ──────────────────────────────────────────────
# Crear tablas en RDS
# ──────────────────────────────────────────────
try:
    # Idempotencia: drop en orden inverso de FK, luego create
    Base.metadata.drop_all(engine)
    Base.metadata.create_all(engine)
    print("✓ Tablas creadas en RDS exitosamente")
except SQLAlchemyError as e:
    print(f"✗ Error al crear tablas: {e}")

✓ Tablas creadas en RDS exitosamente


In [13]:
# ──────────────────────────────────────────────
# 5.4 Inserta los datos con bulk insert SQLAlchemy 2.0
# ──────────────────────────────────────────────
from sqlalchemy import insert
from sqlalchemy.orm import Session

def load_csv(session, model, path, nrows=None):
    """Carga un CSV usando bulk insert de SQLAlchemy 2.0"""
    df = pd.read_csv(path, nrows=nrows)
    # Convertir nombres de columnas a minúsculas para que coincidan con el modelo
    df.columns = df.columns.str.lower()
    # Convertir NaN a None para compatibilidad con PostgreSQL
    records = [
        {k: None if pd.isnull(v) else v for k, v in row.items()}
        for row in df.to_dict(orient="records")
    ]
    # Bulk insert con SQLAlchemy 2.0
    session.execute(insert(model), records)
    print(f"✓ {model.__tablename__}: {len(records):,} filas cargadas")

# Orden respeta dependencias FK:
#   1. airlines (sin dependencias)
#   2. airports (sin dependencias)
#   3. flights (depende de airlines y airports)
with Session(engine) as session:
    load_csv(session, Airlines, "../data/flights/airlines.csv")
    load_csv(session, Airports, "../data/flights/airports.csv")
    load_csv(session, Flights,  "../data/flights/flights.csv", nrows=500_000)  # Solo 500k registros

    session.commit()
    print("✓ Todos los datos cargados y commit realizado")

✓ airlines: 14 filas cargadas
✓ airports: 322 filas cargadas
✓ flights: 500,000 filas cargadas
✓ Todos los datos cargados y commit realizado


In [ ]:
from sqlalchemy import text

verificaciones = {
    "tablas en el schema": """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name
    """,
    "conteo de filas": """
        SELECT 'airlines'  AS tabla, COUNT(*) AS filas FROM airlines  UNION ALL
        SELECT 'airports'  AS tabla, COUNT(*) AS filas FROM airports  UNION ALL
        SELECT 'flights'   AS tabla, COUNT(*) AS filas FROM flights
        ORDER BY tabla
    """,
    "muestra flights (5 filas)": """
        SELECT f.flight_id, f.year, f.month, f.day,
               al.airline AS airline_name,
               f.origin_airport, f.destination_airport,
               f.departure_delay, f.arrival_delay
        FROM flights f
        JOIN airlines al ON f.airline = al.iata_code
        LIMIT 5
    """
}

with engine.connect() as conn:
    for titulo, query in verificaciones.items():
        print(f"\n── {titulo.upper()} ──────────────────────────")
        df = pd.read_sql(text(query), conn)
        print(df.to_string(index=False))